Requested Citation Acknowledgment:
IEEE OTCBVS WS Series Bench; J. Davis and M. Keck, "A two-stage approach to person detection in thermal imagery," In Proc. Workshop on Applications of Computer Vision, January 2005

# Imports

In [ ]:
### Object Detection with Faster R-CNN and RetinaNet in PyTorch - Template Code for Assignment 3 (2026 Spring) ###

### Might need to install the following packages if you haven't already: ###
### You can run these commands in your terminal or uncomment and run them in a notebook cell. ###
# !pip -q install torch torchvision torchmetrics matplotlib
# !pip install pandas
# !pip install pycocotools
# !pip install faster-coco-eval
# !pip install tqdm

import re, random, time
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm


import torch
from torch.utils.data import Dataset, DataLoader


import torchvision
from torchvision.transforms import functional as TF
from torchvision.models.detection import RetinaNet_ResNet50_FPN_V2_Weights, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection import retinanet_resnet50_fpn_v2
from torchvision.models.detection.retinanet import RetinaNetClassificationHead
from torchmetrics.detection.mean_ap import MeanAveragePrecision


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Data Preprocessing and creating data Loaders

In [ ]:
### Helper function for loading and parsing the OSU Thermal Pedestrian Database (OTCBVS) ground-truth files. ###
### Needs to be adapted if you want to use a different dataset. ###


# OSU Thermal Pedestrian Database (OTCBVS) ground-truth parser
# File format:
#   NumberOfFrames
#   img_00001.bmp 4 (x1 y1 x2 y2) (x1 y1 x2 y2) ...
# Notes:
# - Coordinates are pixel indices in the original image (360x240).
# - We'll keep boxes in torchvision format: [x1, y1, x2, y2] (float32)

_BOX_RE = re.compile(r"\((\d+)\s+(\d+)\s+(\d+)\s+(\d+)\)")

def parse_groundtruth_txt(gt_path: str) -> Dict[str, torch.Tensor]:
    with open(gt_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = [
            ln.strip() for ln in f.readlines()
            if ln.strip() and not ln.strip().startswith("%")
        ]

    n_frames = int(lines[0])
    records = lines[1:1 + n_frames]

    out: Dict[str, torch.Tensor] = {}
    for ln in records:
        parts = ln.split()
        fname = parts[0]

        boxes = []
        for m in _BOX_RE.finditer(ln):
            x1, y1, x2, y2 = map(float, m.groups())
            # Safety: ensure proper ordering
            x1, x2 = min(x1, x2), max(x1, x2)
            y1, y2 = min(y1, y2), max(y1, y2)
            boxes.append([x1, y1, x2, y2])

        if boxes:
            out[fname] = torch.tensor(boxes, dtype=torch.float32)
        else:
            out[fname] = torch.zeros((0, 4), dtype=torch.float32)

    return out

# Visualize two samples of the Training Data

# Create the following methods for training and evaluation:
### 1.train_one_epoch
### 2.evaluate_map (mean Average Precision)
### 3.time_inference

In [ ]:
def train_one_epoch(model, loader, optimizer, device, epoch=None):
    ### You Code Here ###



@torch.no_grad()
def evaluate_map(model, loader, device):
    """mAP evaluation using torchmetrics.
    Returns a dict with keys like 'map', 'map_50', 'mar_100', ...
    """
    ### You Code Here ##
    
    return metric.compute()


@torch.no_grad()
def time_inference(model, dataset, device, n_images=100, warmup=10):
    """Average inference time per image (ms)."""
    ### You Code Here ###

    return float(np.mean(times)), float(np.std(times))


# Create Two-stage detector: Faster R-CNN (Region Proposal + classifier)

We'll fine-tune a Faster R-CNN model. To do this you will 
1- Load the fasterRCN.
2- Freeze the backbone.
3- create an optimizer for it. SGD with lr=0.005, momentum=0.9, weight_decay=0.0005

Note: In TorchVision detection models like Faster R-CNN, num_classes means: number of foreground categories + 1 background class

## Train for small number of epochs

# Create One-stage detector: RetinaNet

We'll fine-tune RetinaNet. To do this you will:
1- Load the RetinaNet.
2- Freeze the backbone.
3- create an optimizer for it. SGD with lr=0.005, momentum=0.9, weight_decay=0.0005


Note: In TorchVision detection models, num_classes means: number of foreground categories + 1 background class


## Calculate val_mAP, val_mAP50, mean_infer_ms for both methods

## Visualize the results of two methods for couple of images from test set

## write your conclusion  for the results you got